# 04 · Classical text models

**Question:** Does a small trained text classifier suggest better topic tags than Notebook 03's simple rules?

We compare **two versions of the same learning method**. Both learn from the original titles and the topic tags already assigned in Demo Radar. We change how the title is represented as numbers, then use logistic regression to learn which patterns are associated with each tag. The purpose is to test whether adding letter fragments improves on using words alone. We run the versions separately; a future service would not need both simply because we compared them here.

### What does each version see?

Consider the synthetic title **Klimaschutz und sichere Radwege** (climate protection and safe cycle paths):

| Version | Examples of text features | What we are testing |
|---|---|---|
| Word TF-IDF | Single words such as `klimaschutz` and `radwege`, and word pairs such as `sichere radwege` | How well whole words and short phrases predict tags. |
| Word + character TF-IDF | The same word features, plus fragments of 3–5 characters, such as `rad`, `radw`, and `weg` | Whether shared fragments help with word endings, German compounds, or spelling variations. For example, `Radweg` and `Radwege` share fragments even though they are different whole words. |

**TF-IDF** turns these text features into weighted numbers. A feature receives more weight when it occurs in this title but is relatively uncommon across training titles. The vectorizers learn the vocabulary and frequency weights only from training records. This is a statistical representation of text; it does not itself understand the event. Extra character features might help, add noise, or make little difference. That is why we measure both versions.

### How can one title receive several tags?

**Multi-label** means that several topics can apply at once. Each version fits 12 separate logistic-regression classifiers, one for each approved tag. For a training record, the `mobility` classifier learns whether that tag was assigned; the `environment` classifier learns the same for its own tag, and so on. This is called **one-versus-rest**. At prediction time, both tags can receive high scores; their scores do not need to add up to one. For the synthetic title above, both topics are plausible, but the model's actual suggestions depend on what it learned and the chosen cutoff.

**Historical training labels** means existing production assignments. They let us train without labeling thousands of records again, but some may be wrong. Separately reviewed final labels will test whether the model learned useful topic distinctions rather than merely repeating old mistakes.

### Why keep related demonstrations together?

Suppose the same demonstration series appears on four dates. All four occurrences stay in one saved split: training, validation, or final test. Putting nearly identical editions into both training and testing could make the model look better because it already saw almost the same title. Dates and group IDs control this separation; they are not model features in this notebook.

This is a development experiment. We choose a recommendation threshold on one group-separated part of validation and inspect it on the other part. We also show agreement with all historical validation labels and with the 40 reviewed pilot labels. The pilot has already informed rubric discussions; neither it nor historical agreement is an untouched final-test result. The snapshot loader reads all records to verify the saved split, but this notebook does not inspect, fit, tune, or score the 123 final event-type labels (from 149 reviewed occurrences).

Run top to bottom with the DemoLogic kernel. Private event text and row-level results appear locally; clear notebook outputs before committing. Section 7 saves the main private report and fitted model. Section 8 adds a small optional feature comparison with its own private report.

## 1. Read the fixed development split

The loader verifies the prepared snapshot and benchmark fingerprints. We use only training and validation assignments. `group_id` is used to keep related editions together while dividing validation into threshold-selection and check groups. It is never a model feature.

### Count an event type once (DR-40)

The main comparison uses **one original record per saved event group**: 732 training,
188 validation and 123 final-test representatives. The original 2,626 / 659 / 149
occurrences remain in the snapshot and locked split. We choose the earliest date,
then the lowest event ID if dates tie. In the 40 pilot groups, we keep the record
already selected for review. Neither tags nor model scores choose a record.

We use the selected record's original title and historical tags. If a group has
different title variants or historical tag sets, we retain that fact as a limitation;
we do not combine labels or replace the title with the group's display name.
There are 42 training and four validation groups with conflicting historical tags.
The private selection manifest preserves all member IDs, and every report records
its fingerprint. Earlier occurrence-based reports remain historical experiments
and must not be compared directly with these new results.


In [ ]:
import json
import os
import platform
import sys
from datetime import UTC, datetime
from pathlib import Path
from time import perf_counter

import joblib
import matplotlib.pyplot as plt
import numpy as np
import scipy
import sklearn
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit
from sklearn.multiclass import OneVsRestClassifier
from sklearn.preprocessing import MultiLabelBinarizer

root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents) if (path / "pyproject.toml").is_file()),
    None,
)
if root is None:
    raise FileNotFoundError("Open this notebook inside the DemoLogic repository")
sys.path.insert(0, str(root / "src"))
sys.path.insert(0, str(root))

from notebooks.tools import file_sha256, notebook_sha256, percent
from notebooks.tools import show_table as show
from notebooks.tools import aligned_predictions, comparison_row, matching_report


from demo_logic.benchmark import _csv, load_benchmark, representative_selection
from demo_logic.evaluation import multilabel_metrics, parse_review_tags
from demo_logic.notebook_support import save_private_json
from demo_logic.snapshot import load_snapshot
from demo_logic.taxonomy import topic_tags

In [ ]:
rows, source = load_snapshot(root / "data/dataset.jsonl", root / "data/manifest.json")
bundle = load_benchmark(root / "data/benchmark", rows, source["content_sha256"])
if bundle.manifest.get("status") != "frozen" or source["taxonomy_version"] != "v2":
    raise ValueError("Notebook 04 requires locked split membership and taxonomy v2")

by_id = {row["external_id"]: row for row in rows}
development = {name: [] for name in ("training", "validation")}
for assignment in bundle.manifest["assignments"]:
    if assignment["split"] in development:
        development[assignment["split"]].append(
            {**by_id[assignment["external_id"]], "group_id": assignment["group_id"]}
        )
selection = representative_selection(bundle)
selection_path = save_private_json(
    root / "data/benchmark" / f"{selection['version']}.json", selection
)
selected_ids = {item["external_id"] for item in selection["records"]}
# Keep full validation history only for the separate repeat-event simulation.
validation_occurrences = development["validation"]
training, validation = (
    [row for row in development[name] if row["external_id"] in selected_ids]
    for name in ("training", "validation")
)
assert set(bundle.pilot_ids).issubset({row["external_id"] for row in validation})
tags = sorted(topic_tags(source["taxonomy_version"]))
reviewed_pilot = {
    row["external_id"]: parse_review_tags(row["reviewed_tags"], set(tags))
    for row in _csv(bundle.paths["rubric_pilot"], {"external_id", "reviewed_tags"})
}
if len(reviewed_pilot) != 40:
    raise ValueError("The 40-record development pilot must be complete")

show(
    ["Input", "Count or version"],
    [
        ("Historical training records", len(training)),
        ("Historical validation records", len(validation)),
        ("Reviewed development pilot", len(reviewed_pilot)),
        ("Tag catalog", source["taxonomy_version"]),
        ("Final-test records used for fitting or scoring", 0),
    ],
)

## 2. Turn titles and historical tags into model inputs

Word TF-IDF counts informative single words and word pairs. Character TF-IDF looks at short fragments inside words; it may help with German compounds, spelling variation, and imperfect source text. Both vectorizers learn their vocabulary and frequency weights **only from training titles**. The main comparison keeps the original title text, including any dates written in it. It does not add separate date or location fields, preferred group titles, or reviewer notes. Section 8 later tests a small title-cleaning rule.

### What does “one classifier per tag” mean?

For each title, we ask a separate question for each approved topic: **Does this tag apply?** The training answer is `1` when the historical label includes that tag and `0` when it does not. The same title can have several `1` answers.

For example, suppose the synthetic title **Klimaschutz und sichere Radwege** has the training tags `environment` and `mobility`:

| Classifier's question | Training answer |
|---|---|
| Does `environment` apply? | 1 — yes |
| Does `mobility` apply? | 1 — yes |
| Does `housing` apply? | 0 — no |

`OneVsRestClassifier` fits one logistic regression for each of the 12 approved tags, using the same title features. You run one fitting loop for each feature version; the library handles the 12 tag classifiers. This allows the model to suggest several topics for one demonstration.

For a new title, each classifier produces a **score**, not an immediate yes/no answer. Section 4 applies a cutoff and the maximum-tag policy to turn those scores into suggestions. The historical labels supply the training answers; an absent tag is treated as `0` even if it was missed by the original reviewer. This is why training-label errors and tags with few examples can limit the result.

In [ ]:
MODEL_VERSION = "tfidf-logistic-v1"
RANDOM_SEED = 42
THRESHOLDS = [round(value, 2) for value in np.arange(0.05, 0.81, 0.05)]

train_titles = [row["title"] for row in training]
validation_titles = [row["title"] for row in validation]
encoder = MultiLabelBinarizer(classes=tags)
y_train = encoder.fit_transform([row["tag_slugs"] for row in training])
if list(encoder.classes_) != tags or any(
    y_train[:, index].sum() == 0 for index in range(len(tags))
):
    raise ValueError("A catalog tag has no training examples or the catalog changed")

started = perf_counter()
word_vectorizer = TfidfVectorizer(
    analyzer="word",
    ngram_range=(1, 2),
    min_df=2,
    max_features=30000,
    sublinear_tf=True,
    dtype=np.float64,
)
x_train_word = word_vectorizer.fit_transform(train_titles)
word_train_feature_seconds = perf_counter() - started
started = perf_counter()
x_valid_word = word_vectorizer.transform(validation_titles)
word_valid_feature_seconds = perf_counter() - started

started = perf_counter()
char_vectorizer = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3, 5),
    min_df=2,
    max_features=40000,
    sublinear_tf=True,
    dtype=np.float64,
)
x_train_char = char_vectorizer.fit_transform(train_titles)
char_train_feature_seconds = perf_counter() - started
started = perf_counter()
x_valid_char = char_vectorizer.transform(validation_titles)
char_valid_feature_seconds = perf_counter() - started

feature_sets = {
    "word": (x_train_word, x_valid_word, {"word": word_vectorizer}),
    "word_plus_char": (
        hstack([x_train_word, x_train_char], format="csr"),
        hstack([x_valid_word, x_valid_char], format="csr"),
        {"word": word_vectorizer, "char": char_vectorizer},
    ),
}
show(
    ["Feature set", "Training rows", "Features", "Nonzero training values"],
    [
        (name, matrix.shape[0], matrix.shape[1], matrix.nnz)
        for name, (matrix, _, _) in feature_sets.items()
    ],
)

## 3. Fit both multi-label classifiers

Logistic regression gives each tag a score between zero and one. A higher score means the model learned a stronger association between this title and that tag. These probability estimates have not been **calibrated**: we have not established that their numeric values match observed correctness on independently reviewed cases. Use them to rank tags and apply a tested recommendation cutoff, rather than presenting them as a guaranteed success rate.

The **cutoff** decides which individual tags to suggest. Section 4 chooses it using development data; the recorded run selected 0.25 for word-only and 0.30 for word + character. The separate **80% precision and 80% recall targets** judge the complete set of suggestions against reference answers. They are project requirements, not instructions to set every model's cutoff to 0.80. This explanation does not change the cutoff grid or the agreed quality targets.

The timed fit includes logistic regression but excludes feature construction, which is listed separately. Prediction time includes vectorization plus scoring for the validation batch. No provider API is called, so API spend is zero; local machine and staff costs are not measured here.

In [ ]:
fits = {}
for name, (x_train, x_valid, vectorizers) in feature_sets.items():
    started = perf_counter()
    classifier = OneVsRestClassifier(
        LogisticRegression(solver="liblinear", max_iter=1000, random_state=RANDOM_SEED)
    )
    classifier.fit(x_train, y_train)
    fit_seconds = perf_counter() - started
    started = perf_counter()
    probabilities = classifier.predict_proba(x_valid)
    score_seconds = perf_counter() - started
    if probabilities.shape != (len(validation), len(tags)):
        raise ValueError("Classifier returned an unexpected score matrix")
    fits[name] = {
        "classifier": classifier,
        "vectorizers": vectorizers,
        "probabilities": probabilities,
        "fit_seconds": fit_seconds,
        "score_seconds": score_seconds,
        "feature_count": x_train.shape[1],
    }

show(
    ["Model", "Fit seconds", "Vectorize + score seconds", "Features"],
    [
        (
            name,
            round(run["fit_seconds"], 2),
            round(
                run["score_seconds"]
                + word_valid_feature_seconds
                + (char_valid_feature_seconds if name == "word_plus_char" else 0),
                2,
            ),
            run["feature_count"],
        )
        for name, run in fits.items()
    ],
)

## 4. Choose an output rule without using final-test answers

A suggested tag needs a model score at least as high as the selected threshold. We suggest up to two tags, plus any tag **exactly tied** with the second. We also measure a strict two-tag version. A model can suggest nothing; that lowers coverage and recall rather than hiding a hard case.

### Why does the table show 94 and 94?

The classifiers have already learned from **732 training examples**. We now use the **188 validation examples** for two different jobs:

1. **Choose the cutoff — 94 examples.** Try each cutoff in the fixed grid, from 0.05 to 0.80. For each model and tag-limit policy, keep the cutoff with the highest micro F1 on these examples. If F1 ties, keep the higher cutoff.
2. **Check the chosen cutoff — another 94 examples.** Apply that fixed choice to different event types. Their answers did not help choose the cutoff; their scores show how well it carried over.

These two halves do not overlap. DR-40 kept one representative row per event group, so **94 records represent 94 event types** in each half. We are not counting tags here; one record may have several correct tags. The 123 final-test representatives are a separate set and are not scored in this notebook.

Using different examples for choosing and checking a cutoff reduces the chance of reporting only how well it fits the examples used to choose it. However, we have now inspected these validation results while developing the project. They remain development evidence; the later final test is still needed. The full-validation and 40-record reviewed-pilot tables are additional diagnostics, not new independent tests.

In [ ]:
groups = [row["group_id"] for row in validation]
splitter = GroupShuffleSplit(n_splits=1, test_size=0.5, random_state=RANDOM_SEED)
tune_index, check_index = next(splitter.split(validation_titles, groups=groups))
if {groups[index] for index in tune_index} & {groups[index] for index in check_index}:
    raise AssertionError("Related event types crossed the validation check split")


def select_tags(scores, threshold, *, include_ties=True):
    """Return the highest-scoring approved tags that meet a tested threshold.

    Args:
        scores: One classifier score per tag, in the catalog's fixed order.
        threshold: Minimum score chosen only from development records.
        include_ties: Keep exact ties at the second tag; otherwise cap at two.

    An empty result is an abstention. Slug order breaks equal-score ties so
    rerunning the notebook on the same data produces the same recommendation.
    """
    ranked = sorted(
        ((tag, float(score)) for tag, score in zip(tags, scores) if score >= threshold),
        key=lambda item: (-item[1], item[0]),
    )
    chosen = ranked[:2]
    if include_ties and len(ranked) > 2:
        chosen += [item for item in ranked[2:] if item[1] == chosen[-1][1]]
    return [tag for tag, _ in chosen]


def predictions_at(probabilities, threshold, include_ties):
    """Apply one explicit recommendation policy to every scored event."""
    return [select_tags(scores, threshold, include_ties=include_ties) for scores in probabilities]


def score_indices(reference, predictions, indices):
    """Measure the same event indices under a complete multi-label reference."""
    return multilabel_metrics(
        [reference[index] for index in indices],
        [predictions[index] for index in indices],
        tags,
    )


historical = [row["tag_slugs"] for row in validation]
policies = []
for name, run in fits.items():
    for include_ties in (True, False):
        tried = []
        for threshold in THRESHOLDS:
            predictions = predictions_at(run["probabilities"], threshold, include_ties)
            metrics = score_indices(historical, predictions, tune_index)
            tried.append((threshold, metrics))
        threshold, tune_metrics = max(tried, key=lambda item: (item[1]["micro_f1"] or -1, item[0]))
        predictions = predictions_at(run["probabilities"], threshold, include_ties)
        check_metrics = score_indices(historical, predictions, check_index)
        full_metrics = multilabel_metrics(historical, predictions, tags)
        pilot_indices = [
            index for index, row in enumerate(validation) if row["external_id"] in reviewed_pilot
        ]
        pilot_metrics = multilabel_metrics(
            [reviewed_pilot[validation[index]["external_id"]] for index in pilot_indices],
            [predictions[index] for index in pilot_indices],
            tags,
        )
        policies.append(
            {
                "model": name,
                "policy": "include_cutoff_ties" if include_ties else "strict_two",
                "threshold": threshold,
                "tuning": tune_metrics,
                "group_heldout_check": check_metrics,
                "historical_agreement": full_metrics,
                "reviewed_pilot": pilot_metrics,
                "predictions": predictions,
                "threshold_trials": [
                    {
                        "threshold": value,
                        "micro_precision": result["micro_precision"],
                        "micro_recall": result["micro_recall"],
                        "micro_f1": result["micro_f1"],
                        "coverage": result["coverage"],
                    }
                    for value, result in tried
                ],
            }
        )

show(
    ["Validation purpose", "Demonstrations (rows)", "Distinct event types"],
    [
        (label, len(indices), len({groups[index] for index in indices}))
        for label, indices in [
            ("Choose the cutoff", tune_index),
            ("Check the chosen cutoff", check_index),
        ]
    ],
)

show(
    ["Model", "Tag-limit policy", "Chosen cutoff"],
    [(p["model"], p["policy"], p["threshold"]) for p in policies],
)

### Section 4 summary

We now have one chosen cutoff for each model and tag-limit policy. The first 94 examples chose it; the other 94 check it. A cutoff is a rule for showing a tag, not a claim about overall accuracy. Section 5 measures whether the resulting suggestions are useful enough.

## 5. Read the results against Notebook 03

Precision asks how many suggested tags agree with the reference. Recall asks how many reference tags the model found. Coverage is the share of events with at least one suggestion. The project wants at least **80% precision, 80% recall, and 95% coverage under the same policy**; one good number does not offset another poor one.

The first table uses held-out validation groups and historical labels. The second uses the reviewer’s 40 reviewed pilot examples, which are more carefully labeled but already familiar to the project. For a fair baseline comparison, we load a Notebook 03 report only if its source, split, taxonomy, and pilot fingerprints match this run. Missing or mismatched baseline reports produce an explicit message rather than a misleading comparison.

In [ ]:
columns = ["Candidate", "N", "Precision", "Recall", "F1", "Coverage", "Exact", ">2 tags"]
show(
    columns,
    [
        comparison_row(
            f"{p['model']} / {p['policy']} / t={p['threshold']:.2f}", p["group_heldout_check"]
        )
        for p in policies
    ],
)
print("Reviewed pilot: development diagnostic, not the final test")
show(
    columns,
    [comparison_row(f"{p['model']} / {p['policy']}", p["reviewed_pilot"]) for p in policies],
)

### See the threshold trade-off

This chart uses the threshold-selection groups only. Lower cutoffs can recover more tags but may add wrong suggestions; higher cutoffs can leave demonstrations without a suggestion. The dotted line marks the agreed 80% precision/recall and 95% coverage targets. A later final-test run must check the chosen rule on independently reviewed answers.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for axis, name in zip(axes, fits):
    run = next(p for p in policies if p["model"] == name and p["policy"] == "include_cutoff_ties")
    trials = run["threshold_trials"]
    for metric, label in [
        ("micro_precision", "Precision"),
        ("micro_recall", "Recall"),
        ("coverage", "Coverage"),
    ]:
        axis.plot(
            [item["threshold"] for item in trials],
            [item[metric] if item[metric] is not None else np.nan for item in trials],
            marker=".",
            label=label,
        )
    axis.axvline(run["threshold"], color="gray", linestyle="--", label="Chosen cutoff")
    axis.axhline(0.8, color="black", linestyle=":", linewidth=1)
    axis.axhline(0.95, color="black", linestyle=":", linewidth=1)
    axis.set(title=name.replace("_", " + "), xlabel="Minimum model score", ylim=(0, 1.02))
axes[0].set_ylabel("Fraction of tags or events")
axes[1].legend(loc="lower left")
fig.suptitle("Validation threshold-selection groups; historical labels")
fig.tight_layout()
plt.show()
plt.close(fig)

In [ ]:
expected_fingerprints = {
    str(selection_path.relative_to(root)): file_sha256(selection_path),
    "data/manifest.json": file_sha256(root / "data/manifest.json"),
    "data/benchmark/split-manifest.json": file_sha256(bundle.paths["manifest"]),
    "data/benchmark/rubric-pilot-queue.csv": file_sha256(bundle.paths["rubric_pilot"]),
}
matched = matching_report(
    root / "reports/baselines",
    "baseline_version",
    source=source,
    selection=selection,
    fingerprints=expected_fingerprints,
    required=False,
)
baseline = matched[1] if matched else None
if matched:
    print("Matched private Notebook 03 report:", matched[0].name)

if baseline is None:
    print("No matching Notebook 03 report. Run that notebook on this snapshot for comparison.")
else:
    baseline_rows = []
    for run in baseline["runs"]:
        if run["candidate"] not in {"most_common", "keywords"}:
            continue
        predicted = aligned_predictions(run["predictions"], validation)
        metrics = score_indices(historical, predicted, check_index)
        baseline_rows.append(comparison_row(f"{run['candidate']} / {run['policy']}", metrics))
    show(
        columns,
        baseline_rows
        + [
            comparison_row(f"{p['model']} / {p['policy']}", p["group_heldout_check"])
            for p in policies
        ],
    )

### Section 5 summary — what did we learn?

**Recorded checkpoint: 28 September 2026.** The table below uses the same 94 check examples and their historical labels, with up to two tags plus exact ties. The executable tables above show the current run; revisit this written interpretation if the data or settings change.

| Candidate | Precision | Recall | F1 | Coverage |
|---|---:|---:|---:|---:|
| Most-common-tag baseline | 26.6% | 19.7% | 22.6% | 100.0% |
| Regex baseline | 78.4% | 54.3% | 64.2% | 69.1% |
| Words + word pairs | 55.3% | 49.6% | 52.3% | 81.9% |
| Words + word pairs + character fragments | 71.7% | 52.0% | 60.3% | 73.4% |

**Character fragments help within the trained-model comparison.** They raise precision and F1 compared with word features alone, but fewer demonstrations receive a suggestion. Both are still below the required 80% precision, 80% recall, and 95% coverage.

**The trained models have not beaten the regex baseline on F1 in this check.** Regex achieves 64.2% F1 versus 60.3% for words plus characters, although its coverage is lower. This is agreement with historical production tags, which may contain errors; it does not establish a final winner against independently reviewed answers.

**The reviewed pilot also shows substantial room for improvement.** On the reviewer’s 40 reviewed development examples, words plus characters achieve 66.7% precision, 36.1% recall, and 70.0% coverage. Better performance against historical labels does not remove the need to check label quality and missing context.

**Takeaway:** Keep these as measured baselines for the next candidate. None of the shown methods meets all three requirements. The two tag-limit policies produced the same results for the trained models in this run; allowing exact ties did not change their suggestions.

## 6. Check weak tags and actual mistakes

The table below shows support and errors for each tag in the **separate validation check**. Small support means a high or low percentage may rest on very few examples. The next table shows the first eight check-set disagreements by ID. These are private examples for diagnosis, not cherry-picked successes and not instructions to change human labels to match a model.

In [ ]:
chosen = max(
    policies,
    key=lambda p: (
        p["tuning"]["micro_f1"] or -1,
        p["tuning"]["coverage"],
        p["model"],
        p["policy"] == "include_cutoff_ties",
    ),
)
print("Development candidate:", chosen["model"], chosen["policy"], "threshold", chosen["threshold"])
show(
    ["Tag", "Reference events", "Correct", "Extra", "Missed", "Precision", "Recall"],
    [
        (
            tag,
            score["support"],
            score["tp"],
            score["fp"],
            score["fn"],
            percent(score["precision"]),
            percent(score["recall"]),
        )
        for tag, score in chosen["group_heldout_check"]["per_tag"].items()
    ],
)
mistakes = []
for index in sorted(check_index, key=lambda index: validation[index]["external_id"]):
    row = validation[index]
    expected, actual = set(row["tag_slugs"]), set(chosen["predictions"][index])
    if expected != actual:
        mistakes.append(
            (
                row["external_id"],
                row["title"],
                ", ".join(sorted(expected - actual)),
                ", ".join(sorted(actual - expected)),
            )
        )
show(["ID", "Original title", "Missed", "Extra"], mistakes[:8])

### Section 6 insight — an overall score can hide missed topics

In the 28 September check, the words-plus-characters candidate finds only **1 of 11 housing labels** and **1 of 10 social-issues labels**. These are concrete weaknesses to inspect in later comparisons. There are no `religion` reference labels in this 94-record check, so it cannot tell us how well the model recognizes that topic.

Read each tag's support alongside its score. A strong result on one or two examples is weak evidence. The displayed disagreements help us investigate possible model errors, missing context, or historical-label errors; they do not justify changing reference answers to agree with predictions.

## 7. Save the private experiment record

We save the selected fitted model for later local checking, plus scores and row-level predictions for all policies in a private JSON report. They are **not public Git artifacts**. The report records the exact input fingerprints, scikit-learn version, settings, threshold search, timings, and sample limitations. Selection here gives a candidate for further comparison; it does not approve deployment. If none of the methods meets the final reviewed-set requirements, keeping manual review is an acceptable conclusion.

In [ ]:
run_id = datetime.now(UTC).strftime("%Y%m%dT%H%M%S%fZ")
notebook_path = root / "notebooks/04_classical_text_models.ipynb"
notebook_source_sha256 = notebook_sha256(notebook_path)
report = {
    "run_id": run_id,
    "model_version": MODEL_VERSION,
    "selection_version": selection["version"],
    "selection_sha256": selection["selection_sha256"],
    "notebook_source_sha256": notebook_source_sha256,
    "snapshot_sha256": source["content_sha256"],
    "taxonomy_version": source["taxonomy_version"],
    "fingerprints": expected_fingerprints,
    "python": platform.python_version(),
    "sklearn": sklearn.__version__,
    "scipy": scipy.__version__,
    "training_records": len(training),
    "validation_records": len(validation),
    "threshold_selection_records": len(tune_index),
    "group_heldout_check_records": len(check_index),
    "threshold_grid": THRESHOLDS,
    "random_seed": RANDOM_SEED,
    "feature_settings": {
        "word": {
            "analyzer": "word",
            "ngram_range": [1, 2],
            "min_df": 2,
            "max_features": 30000,
            "sublinear_tf": True,
            "dtype": "float64",
        },
        "char": {
            "analyzer": "char_wb",
            "ngram_range": [3, 5],
            "min_df": 2,
            "max_features": 40000,
            "sublinear_tf": True,
            "dtype": "float64",
        },
    },
    "feature_seconds": {
        "word_train": word_train_feature_seconds,
        "word_validation": word_valid_feature_seconds,
        "char_train": char_train_feature_seconds,
        "char_validation": char_valid_feature_seconds,
    },
    "fit_seconds": {name: run["fit_seconds"] for name, run in fits.items()},
    "score_seconds": {name: run["score_seconds"] for name, run in fits.items()},
    "feature_counts": {name: run["feature_count"] for name, run in fits.items()},
    "selected_candidate": {
        "model": chosen["model"],
        "policy": chosen["policy"],
        "threshold": chosen["threshold"],
    },
    "api_cost_usd": 0,
    "local_compute_cost_measured": False,
    "limitations": [
        "Training and most validation references are historical assignments, not independently reviewed labels.",
        "The 40 reviewed pilot records informed rubric work and are development diagnostics only.",
        "The internal check used different validation groups, but repeated candidate inspection may still bias selection.",
        "Final-test records were loaded for split verification but not inspected, fitted, tuned, or scored.",
        "Rare tags may have too little support for reliable per-tag conclusions.",
    ],
    "policies": [
        {
            **{key: value for key, value in policy.items() if key != "predictions"},
            "predictions": [
                {
                    "external_id": row["external_id"],
                    "predicted": prediction,
                    "scores": dict(
                        zip(tags, map(float, fits[policy["model"]]["probabilities"][index]))
                    ),
                }
                for index, (row, prediction) in enumerate(zip(validation, policy["predictions"]))
            ],
        }
        for policy in policies
    ],
}
artifact_dir = root / "artifacts/classical"
artifact_dir.mkdir(parents=True, exist_ok=True, mode=0o700)
artifact_path = artifact_dir / f"{run_id}.joblib"
descriptor = os.open(artifact_path, os.O_WRONLY | os.O_CREAT | os.O_EXCL, 0o600)
with os.fdopen(descriptor, "wb") as destination:
    joblib.dump(
        {
            "model_version": MODEL_VERSION,
            "selection_version": selection["version"],
            "selection_sha256": selection["selection_sha256"],
            "catalog_tags": tags,
            "model": fits[chosen["model"]]["classifier"],
            "vectorizers": fits[chosen["model"]]["vectorizers"],
            "recommendation_policy": report["selected_candidate"],
            "source_snapshot_sha256": source["content_sha256"],
        },
        destination,
    )
report["model_file_bytes"] = artifact_path.stat().st_size
report_path = save_private_json(root / "reports/classical" / f"{run_id}.json", report)
print("Private report:", report_path)
print("Private fitted model:", artifact_path)
print("Model file bytes:", artifact_path.stat().st_size)

## 8. Optional feature-engineering comparison (DR-41)

We had time in Week 2 to test whether small changes to title features help before adding a more complex model. Word pairs are **already included** in the main word model. Here we compare two changes against that fixed reference; each changes only one feature choice.

| Candidate | Change from the existing word model | Synthetic example |
|---|---|---|
| Existing words + pairs | Reference from sections 2–5 | `sichere`, `radwege`, and `sichere radwege` |
| Single words only | Remove word pairs; keep the original title | `sichere` and `radwege` |
| Words + pairs, schedule removed | Keep word pairs; remove a narrowly defined trailing schedule bracket | `Sichere Radwege (vom 02.10.2025 bis 25.06.2026 - jeweils Do.)` becomes `Sichere Radwege` |

The cleaner recognizes only a trailing bracket containing a numeric date range, optionally followed by one weekday. It keeps topic brackets such as `(gegen Rassismus)`, years used within a title, and schedule formats it does not recognize. This deliberately small rule is easy to inspect; it does not rewrite the source data. We show how many training and validation titles it actually changes.

All candidates use the same 732 training and 188 validation representatives, logistic regression, threshold grid, and recommendation policy (up to two tags plus exact ties). Each candidate chooses its cutoff using the same 94 threshold-selection records. We show results on the same 94 check records, using historical labels. Because we have already inspected this check set, these are **exploratory development results**, not fresh evidence of final-test performance. No final-test labels are used.

The report records precision, recall, F1, coverage, feature count, preparation/fit time, and validation-batch prediction time. API spend is zero; local compute is not priced. These experiments do not automatically replace the model saved in section 7.

In [ ]:
import re

FEATURE_EXPERIMENT_VERSION = "title-features-v1"
# Only a full trailing date-range bracket, optionally with one named weekday.
SCHEDULE_SUFFIX = (
    r"\s*\((?:vom\s+)?\d{1,2}\.\d{1,2}\.\d{4}\s*(?:bis|[-–])\s*"
    r"\d{1,2}\.\d{1,2}\.\d{4}"
    r"(?:\s*[-,]\s*(?:jeweils\s+)?"
    r"(?:Mo(?:ntag)?|Di(?:enstag)?|Mi(?:ttwoch)?|Do(?:nnerstag)?|"
    r"Fr(?:eitag)?|Sa(?:mstag)?|So(?:nntag)?)\.?)?\)\s*$"
)
schedule_pattern = re.compile(SCHEDULE_SUFFIX, flags=re.IGNORECASE)


def remove_schedule_suffix(title):
    """Remove one recognized trailing schedule bracket, keeping the topic text.

    Unknown formats and topic brackets stay unchanged. If removal would leave
    no title, keep the original. The returned text is an experiment input;
    the snapshot and saved review titles are never edited.
    """
    match = schedule_pattern.search(title)
    if match is None:
        return title
    return title[: match.start()].rstrip() or title


synthetic_titles = [
    "Sichere Radwege (vom 02.10.2025 bis 25.06.2026 - jeweils Do.)",
    "Demonstration (gegen Rassismus)",
    "Erinnerung an 1989",
]
show(
    ["Synthetic input", "After schedule cleaning"],
    [(title, remove_schedule_suffix(title)) for title in synthetic_titles],
)

started = perf_counter()
clean_train = [remove_schedule_suffix(title) for title in train_titles]
clean_train_seconds = perf_counter() - started
started = perf_counter()
clean_validation = [remove_schedule_suffix(title) for title in validation_titles]
clean_validation_seconds = perf_counter() - started
changed_counts = {
    "training": sum(a != b for a, b in zip(train_titles, clean_train)),
    "validation": sum(a != b for a, b in zip(validation_titles, clean_validation)),
}
show(["Split", "Titles with a removed schedule"], changed_counts.items())

feature_variants = {
    "single_words": (train_titles, validation_titles, (1, 1), 0.0, 0.0),
    "schedule_removed": (
        clean_train,
        clean_validation,
        (1, 2),
        clean_train_seconds,
        clean_validation_seconds,
    ),
}
feature_trials = {}
for name, (train_text, valid_text, ngrams, prep_train, prep_valid) in feature_variants.items():
    # Copy the reference settings; only ngram_range or the input text changes.
    vectorizer = TfidfVectorizer(**{**word_vectorizer.get_params(), "ngram_range": ngrams})
    started = perf_counter()
    x_train = vectorizer.fit_transform(train_text)
    train_feature_seconds = perf_counter() - started + prep_train
    started = perf_counter()
    x_valid = vectorizer.transform(valid_text)
    valid_feature_seconds = perf_counter() - started + prep_valid
    classifier = OneVsRestClassifier(
        LogisticRegression(solver="liblinear", max_iter=1000, random_state=RANDOM_SEED)
    )
    started = perf_counter()
    classifier.fit(x_train, y_train)
    fit_seconds = perf_counter() - started
    started = perf_counter()
    scores = classifier.predict_proba(x_valid)
    prediction_seconds = perf_counter() - started + valid_feature_seconds
    feature_trials[name] = {
        "probabilities": scores,
        "feature_count": x_train.shape[1],
        "ngram_range": list(ngrams),
        "train_feature_seconds": train_feature_seconds,
        "fit_seconds": fit_seconds,
        "validation_prediction_seconds": prediction_seconds,
    }

# Reuse the existing reference fit, so the comparison does not depend on a refit.
feature_trials["words_plus_pairs"] = {
    "probabilities": fits["word"]["probabilities"],
    "feature_count": fits["word"]["feature_count"],
    "ngram_range": [1, 2],
    "train_feature_seconds": word_train_feature_seconds,
    "fit_seconds": fits["word"]["fit_seconds"],
    "validation_prediction_seconds": word_valid_feature_seconds + fits["word"]["score_seconds"],
}

In [ ]:
feature_results = []
for name in ("words_plus_pairs", "single_words", "schedule_removed"):
    trial = feature_trials[name]
    threshold_trials = []
    for threshold in THRESHOLDS:
        predictions = predictions_at(trial["probabilities"], threshold, include_ties=True)
        metrics = score_indices(historical, predictions, tune_index)
        threshold_trials.append({"threshold": threshold, "metrics": metrics})
    selected = max(
        threshold_trials,
        key=lambda item: (
            item["metrics"]["micro_f1"] if item["metrics"]["micro_f1"] is not None else -1,
            item["threshold"],
        ),
    )
    predictions = predictions_at(trial["probabilities"], selected["threshold"], include_ties=True)
    feature_results.append(
        {
            "candidate": name,
            "threshold": selected["threshold"],
            "tuning": selected["metrics"],
            "check": score_indices(historical, predictions, check_index),
            "threshold_trials": threshold_trials,
            **{key: value for key, value in trial.items() if key != "probabilities"},
            "predictions": [
                {"external_id": row["external_id"], "tags": prediction}
                for row, prediction in zip(validation, predictions)
            ],
        }
    )

show(
    [
        "Candidate",
        "Cutoff",
        "Precision",
        "Recall",
        "F1",
        "Coverage",
        "Features",
        "Prepare + fit (s)",
        "Predict 188 titles (s)",
    ],
    [
        (
            result["candidate"],
            result["threshold"],
            *[
                percent(result["check"][key])
                for key in ("micro_precision", "micro_recall", "micro_f1", "coverage")
            ],
            result["feature_count"],
            round(result["train_feature_seconds"] + result["fit_seconds"], 3),
            round(result["validation_prediction_seconds"], 3),
        )
        for result in feature_results
    ],
)
reference = feature_results[0]["check"]
for result in feature_results:
    metrics = result["check"]
    meets_targets = all(
        metrics[key] is not None and metrics[key] >= target
        for key, target in (("micro_precision", 0.8), ("micro_recall", 0.8), ("coverage", 0.95))
    )
    delta = metrics["micro_f1"] - reference["micro_f1"]
    print(
        f"{result['candidate']}: F1 change {100 * delta:+.1f} percentage points; "
        f"meets all three development targets: {meets_targets}"
    )

### How to read this experiment

Compare **single words** with **words + pairs** to judge whether phrases add value. Compare **schedule removed** with **words + pairs** to judge only the cleaning rule. Do not credit a result to both changes at once. Read precision, recall, and coverage together: a higher F1 alone does not mean all project targets are met. Timings are from one local run and can change with machine load.

Check the changed-title counts before interpreting the cleaning result. If few titles changed, the evidence concerns only this narrow rule. We have not tested every way to remove scheduling text, nor added location, time, or nearby-place features.

Record the lesson even if the simpler feature choice wins or none improves on the reference. Keep the final answers out of this decision. This section saves its own experiment report, linked to the original run, for later comparison and presentation review.

In [ ]:
feature_report = {
    "experiment_version": FEATURE_EXPERIMENT_VERSION,
    "run_id": run_id,
    "base_report": str(report_path.relative_to(root)),
    "notebook_source_sha256": notebook_source_sha256,
    "snapshot_sha256": source["content_sha256"],
    "selection_sha256": selection["selection_sha256"],
    "taxonomy_version": source["taxonomy_version"],
    "fingerprints": expected_fingerprints,
    "python": platform.python_version(),
    "sklearn": sklearn.__version__,
    "random_seed": RANDOM_SEED,
    "threshold_grid": THRESHOLDS,
    "recommendation_policy": "include_cutoff_ties",
    "base_word_settings": report["feature_settings"]["word"],
    "classifier_settings": {"solver": "liblinear", "max_iter": 1000, "random_state": RANDOM_SEED},
    "schedule_pattern": SCHEDULE_SUFFIX,
    "schedule_pattern_flags": ["IGNORECASE"],
    "changed_titles": changed_counts,
    "training_ids": [row["external_id"] for row in training],
    "threshold_selection_ids": [validation[index]["external_id"] for index in tune_index],
    "check_ids": [validation[index]["external_id"] for index in check_index],
    "final_test_records_scored": 0,
    "api_cost_usd": 0,
    "local_compute_cost_measured": False,
    "limitations": [
        "Exploratory comparison on previously inspected development data with historical labels.",
        "Only one narrow schedule format is removed; topic text and source rows are preserved.",
        "Single-run timings are not a production latency benchmark.",
        "No automatic change to the selected artifact; final-test labels were not used for fitting, tuning or scoring.",
    ],
    "results": feature_results,
}
feature_report_path = save_private_json(
    root / "reports/classical" / f"{run_id}-feature-engineering.json", feature_report
)
print("Private feature-engineering report:", feature_report_path)

## 9. Experiment conclusions and next step

**Checkpoint and accepted decision:**

- **Main comparison:** Adding character fragments improves F1 from 52.3% to 60.3% over word features on the 94 historical-label check examples. Coverage falls from 81.9% to 73.4%. The regex baseline still has higher F1 on these examples.
- **Extra feature experiments:** Single words achieve 50.2% F1 and schedule cleaning 51.5%, compared with 52.3% for the existing words-plus-pairs reference. Single words increase coverage but lose precision. Cleaning changes only four training titles and two validation titles, so its result says little about broader cleaning methods.
- **Decision accepted:** Keep the existing word and word-plus-character comparison. Do not promote either extra feature variant from section 8. Keep their reports as evidence of what we tried and why we stopped.
- **Limits:** None of the compared methods meets all three targets. Most results use historical labels; the reviewed pilot is small and already familiar. The final-test answers have not been used for fitting, tuning, or scoring.
- **Next step:** Compare the planned semantic candidate using the same development setup. Preserve the selected classical candidate and regex baseline for the later final comparison once candidate settings are settled.

The private reports saved in sections 7 and 8 contain settings, fingerprints, timings, and predictions. The [Week 2 plan](../docs/methodology.md#comparison-stages) records the decision, and the [insights log](../docs/presentation/insights.md) keeps the presentation lessons.